In [1]:
from pathlib import Path
import sys


sys.path.append(str(Path().resolve().parent))

from utilities.src import build_df



  



In [2]:
BASE_DIR = Path().resolve()
PROJECT_ROOT = BASE_DIR.parent   #to go un up the the root
DATA_DIR = PROJECT_ROOT / "data"

# Root directory of dataset
root = Path(DATA_DIR)


In [3]:
templates_df=build_df(root)

value_cols = [
    'acc_x', 'acc_y', 'acc_z',
    'gyr_x', 'gyr_y', 'gyr_z',
    'mag_x', 'mag_y', 'mag_z'
]

merged_df = (
    templates_df
    .pivot_table(
        index=['subject', 'exercise', 'time index', 'execution_type'],
        columns='unit',
        values=value_cols,
        aggfunc='first'
    )
)

merged_df.columns = [f'{feature}_{unit}' for feature, unit in merged_df.columns]
merged_df = merged_df.reset_index()

merged_df.head(300)

,subject,exercise,time index,execution_type,acc_x_u1,acc_x_u2,acc_x_u3,acc_x_u4,acc_x_u5,acc_y_u1,...,mag_y_u1,mag_y_u2,mag_y_u3,mag_y_u4,mag_y_u5,mag_z_u1,mag_z_u2,mag_z_u3,mag_z_u4,mag_z_u5
0,s1,e1,314,correct,-9.675384,-9.623113,-3.064462,9.092290,-4.256507,-1.665096,...,0.457257,-0.485758,-0.794563,0.001059,0.733553,-0.081429,0.005508,-0.097777,-0.546160,0.216855
1,s1,e1,315,correct,-9.660426,-9.622773,-3.064653,9.077757,-4.254075,-1.687629,...,0.457905,-0.489491,-0.794913,0.001092,0.732285,-0.081475,0.005818,-0.096094,-0.547180,0.215713
2,s1,e1,316,correct,-9.660451,-9.681846,-3.049775,9.057949,-4.251647,-1.702569,...,0.457524,-0.493425,-0.794522,0.000979,0.732651,-0.081558,0.004581,-0.096575,-0.547338,0.216238
3,s1,e1,317,correct,-9.660356,-9.554703,-3.064489,9.163527,-4.249214,-1.657827,...,0.457243,-0.497159,-0.795803,0.002048,0.732930,-0.080656,0.004653,-0.096933,-0.548436,0.216331
4,s1,e1,318,correct,-9.645445,-9.404443,-3.086412,9.094555,-4.261387,-1.687700,...,0.457522,-0.501722,-0.795687,0.000966,0.731635,-0.081974,0.002505,-0.097586,-0.547134,0.216355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,s1,e1,1091,fast,-9.649157,-4.945461,-3.085121,8.945818,-4.042970,-1.693186,...,0.458859,-0.764789,-0.779178,0.010510,0.724770,-0.063939,-0.131169,-0.161052,-0.565352,0.235095
296,s1,e1,1092,fast,-9.671609,-5.170185,-2.942373,9.014229,-4.116072,-1.693157,...,0.459247,-0.768058,-0.779974,0.010659,0.724566,-0.064109,-0.130146,-0.160978,-0.565843,0.232137
297,s1,e1,1093,fast,-9.664093,-5.647275,-2.860831,9.043826,-4.145386,-1.678253,...,0.459261,-0.766568,-0.779512,0.008947,0.725329,-0.065017,-0.123659,-0.161971,-0.565781,0.230639
298,s1,e1,1094,fast,-9.656646,-6.244500,-3.010625,8.994843,-4.106359,-1.708195,...,0.459276,-0.764337,-0.781452,0.008957,0.726310,-0.065925,-0.119948,-0.162519,-0.565519,0.230824


In [6]:
import pandas as pd
 
 
def analyze_segment_lengths(templates_df):
    """
    For each sample_id, count the number of timesteps.
    Then summarize by execution_type: mean, std, min, max.
    """
 
    # Step 1: count timesteps per sample
    lengths = (
        templates_df
        .groupby([ "execution_type", "subject", "exercise"])
        .size()
        .reset_index(name="n_timesteps")
    )
 
    # Step 2: summary stats grouped by execution type
    summary = (
        lengths
        .groupby("execution_type")["n_timesteps"]
        .agg(["mean", "std", "min", "max", "count"])
        .round(1)
    )
 
    print("=== Timesteps per execution type ===")
    print(summary)
    print()
 
    # Step 3: per exercise breakdown (to see if pattern holds across all 8)
    per_exercise_mean = (
        lengths
        .groupby(["exercise", "execution_type"])["n_timesteps"]
        .mean()
        .round(1)
        .unstack("execution_type")
    )
    per_exercise_mean.columns = [f"{c}_mean" for c in per_exercise_mean.columns]
 
    per_exercise_max = (
        lengths
        .groupby(["exercise", "execution_type"])["n_timesteps"]
        .max()
        .unstack("execution_type")
    )
    per_exercise_max.columns = [f"{c}_max" for c in per_exercise_max.columns]
 
    per_exercise_min = (
        lengths
        .groupby(["exercise", "execution_type"])["n_timesteps"]
        .min()
        .unstack("execution_type")
    )
    per_exercise_min.columns = [f"{c}_min" for c in per_exercise_min.columns]
 
    per_exercise = pd.concat([per_exercise_mean, per_exercise_max, per_exercise_min], axis=1).sort_index(axis=1)
 
    print("=== Mean, min and max timesteps per exercise × execution type ===")
    print(per_exercise)
 
    return lengths, summary, per_exercise

In [7]:
lengths, summary, per_exercise = analyze_segment_lengths(merged_df)

=== Timesteps per execution type ===
                 mean   std  min  max  count
execution_type                              
correct         142.3  30.9   98  218     40
fast             71.8  17.4   49  116     40
low_amplitude   125.2  27.5   84  203     40

=== Mean, min and max timesteps per exercise × execution type ===
          correct_max  correct_mean  correct_min  fast_max  fast_mean  \
exercise                                                                
e1                215         170.4          146       113       84.2   
e2                200         158.0          137        91       79.6   
e3                168         143.6          121       113       80.6   
e4                159         128.2           98        65       58.8   
e5                148         127.2           98        75       62.0   
e6                218         142.6          114       116       78.8   
e7                176         136.0           99        73       65.6   
e8            

In [9]:
sample_x = templates_df[templates_df["sample_id"] == 301]
print(len(sample_x))                          
print(sample_x["execution_type"].iloc[0])    
print(sample_x["time index"].min(), "→", sample_x["time index"].max())

52
fast
531 → 582


In [12]:
def compare_sensor_means(templates_df, exec_types=("correct", "low_amplitude")):
    """
    Compare average sensor readings between two execution types,
    broken down by exercise × unit combination.
 
    For each (exercise, unit) pair and each sensor group (acc, gyr, mag),
    computes the mean per axis per execution type, plus the diff row.
 
    Returns a dict: { (exercise, unit): { sensor: DataFrame } }
    """
 
    sensor_cols = {
        "acc": ["acc_x", "acc_y", "acc_z"],
        "gyr": ["gyr_x", "gyr_y", "gyr_z"],
        "mag": ["mag_x", "mag_y", "mag_z"],
    }
 
    subset = templates_df[templates_df["execution_type"].isin(exec_types)]
 
    results = {}
 
    for (exercise, unit), group in subset.groupby(["exercise", "unit"]):
        key = (exercise, unit)
        results[key] = {}
 
        print(f"\n{'='*50}")
        print(f"Exercise: {exercise}  |  Unit: {unit}")
        print(f"{'='*50}")
 
        for sensor, cols in sensor_cols.items():
            sensor_means = (
                group
                .groupby("execution_type")[cols]
                .mean()
                .round(4)
            )
 
            # diff row: second exec_type minus first
            if all(t in sensor_means.index for t in exec_types):
                sensor_means.loc[f"diff ({exec_types[1]} - {exec_types[0]})"] = (
                    sensor_means.loc[exec_types[1]] - sensor_means.loc[exec_types[0]]
                )
 
            results[key][sensor] = sensor_means
 
            print(f"\n  {sensor.upper()}")
            print(sensor_means.to_string())
 
    return results

In [13]:
results = compare_sensor_means(templates_df)


Exercise: e1  |  Unit: u1

  ACC
                                 acc_x   acc_y   acc_z
execution_type                                        
correct                        -9.5288 -1.6244 -0.2578
low_amplitude                  -9.5456 -1.5523 -0.3465
diff (low_amplitude - correct) -0.0168  0.0721 -0.0887

  GYR
                                 gyr_x   gyr_y   gyr_z
execution_type                                        
correct                        -0.0074  0.0031 -0.0041
low_amplitude                  -0.0070  0.0030 -0.0041
diff (low_amplitude - correct)  0.0004 -0.0001  0.0000

  MAG
                                 mag_x   mag_y   mag_z
execution_type                                        
correct                         0.5806  0.4694  0.0032
low_amplitude                   0.5851  0.4656  0.0128
diff (low_amplitude - correct)  0.0045 -0.0038  0.0096

Exercise: e1  |  Unit: u2

  ACC
                                 acc_x   acc_y   acc_z
execution_type                        

In [14]:
results[("e2", "u5")]

{'acc':                                  acc_x   acc_y   acc_z
 execution_type                                        
 correct                        -3.2656 -8.6821 -0.2792
 low_amplitude                  -2.5998 -8.8514 -0.1379
 diff (low_amplitude - correct)  0.6658 -0.1693  0.1413,
 'gyr':                                  gyr_x   gyr_y   gyr_z
 execution_type                                        
 correct                         0.0042 -0.0012 -0.0098
 low_amplitude                   0.0011 -0.0040 -0.0082
 diff (low_amplitude - correct) -0.0031 -0.0028  0.0016,
 'mag':                                  mag_x   mag_y   mag_z
 execution_type                                        
 correct                        -0.1051  0.7114  0.1058
 low_amplitude                  -0.1626  0.7010  0.0831
 diff (low_amplitude - correct) -0.0575 -0.0104 -0.0227}

In [15]:
def flag_differences(results, negligible=0.02, noticeable=0.05):
    """
    Takes the output of compare_sensor_means and flags axes where
    the diff between execution types is noticeable.
 
    Uses a relative threshold: diff% = |diff| / |correct_mean| × 100
    so the comparison is fair across sensors with different scales.
 
    Thresholds (as fractions):
        < negligible          → negligible (noise)
        negligible – noticeable → marginal
        > noticeable          → noticeable
 
    Prints a summary and returns a DataFrame of flagged axes only.
    """
 
    rows = []
 
    for (exercise, unit), sensor_dict in results.items():
        for sensor, df in sensor_dict.items():
 
            # diff row is always the last row
            diff_row = df.iloc[-1]
            correct_row = df.loc["correct"] if "correct" in df.index else df.iloc[0]
 
            for col in diff_row.index:
                abs_diff = abs(diff_row[col])
                base = abs(correct_row[col])
 
                # avoid division by zero
                rel_diff = (abs_diff / base) if base > 1e-6 else 0.0
 
                if rel_diff < negligible:
                    flag = "negligible"
                elif rel_diff < noticeable:
                    flag = "marginal"
                else:
                    flag = "NOTICEABLE"
 
                rows.append({
                    "exercise": exercise,
                    "unit": unit,
                    "sensor": sensor,
                    "axis": col,
                    "correct_mean": round(correct_row[col], 4),
                    "diff": round(diff_row[col], 4),
                    "diff_%": round(rel_diff * 100, 2),
                    "flag": flag,
                })
 
    flagged_df = pd.DataFrame(rows)
 
    # Summary count
    counts = flagged_df["flag"].value_counts()
    print("=== Flag summary ===")
    print(counts.to_string())
    print()
 
    # Print only noticeable ones
    noticeable_df = flagged_df[flagged_df["flag"] == "NOTICEABLE"].sort_values(
        ["exercise", "unit", "sensor"]
    )
    print(f"=== NOTICEABLE differences (>{noticeable*100:.0f}% relative diff) ===")
    if noticeable_df.empty:
        print("None found.")
    else:
        print(noticeable_df.to_string(index=False))
 
    return flagged_df

In [16]:

flagged_df = flag_differences(results)

=== Flag summary ===
flag
NOTICEABLE    262
negligible     49
marginal       49

=== NOTICEABLE differences (>5% relative diff) ===
exercise unit sensor  axis  correct_mean    diff  diff_%       flag
      e1   u1    acc acc_z       -0.2578 -0.0887   34.41 NOTICEABLE
      e1   u1    gyr gyr_x       -0.0074  0.0004    5.41 NOTICEABLE
      e1   u1    mag mag_z        0.0032  0.0096  300.00 NOTICEABLE
      e1   u2    acc acc_x       -3.2707 -2.3260   71.12 NOTICEABLE
      e1   u2    acc acc_y        7.8141 -0.7778    9.95 NOTICEABLE
      e1   u2    acc acc_z        0.1536  0.1518   98.83 NOTICEABLE
      e1   u2    gyr gyr_x        0.0001  0.0024 2400.00 NOTICEABLE
      e1   u2    gyr gyr_y        0.0244  0.0020    8.20 NOTICEABLE
      e1   u2    gyr gyr_z       -0.0105 -0.0006    5.71 NOTICEABLE
      e1   u2    mag mag_x       -0.1034  0.2012  194.58 NOTICEABLE
      e1   u2    mag mag_y       -0.6515 -0.0479    7.35 NOTICEABLE
      e1   u2    mag mag_z        0.0775 -0.0113   1